# 01 · Ingest Liiga data

We download six seasons of games from the **public liiga.fi v2 API**
(`/api/v2/games`) — five to learn from (2021-22 … 2025-26) and the **2026-27**
schedule we want to predict. liiga.fi numbers seasons by their end year, so
`season=2027` is 2026-27.

Each game carries goal events (scorer + assist player IDs), which is how we
later measure player production. We also harvest player bios (date of birth,
position) from one game per team, for the age curve and positional priors.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

In [ ]:
from liiga.ingest import ingest_all, harvest_bios
counts = ingest_all()           # cached on disk after the first run
counts

In [ ]:
print('player bios captured:', harvest_bios())

In [ ]:
from liiga.db import get_connection, query_df
con = get_connection()
display(query_df(con, '''SELECT season, COUNT(*) games,
                                SUM(ended::INT) played
                         FROM raw_games GROUP BY season ORDER BY season'''))
con.close()

**Check:** the 2027 row should show games scheduled but `played = 0` — that is
the season we forecast. Training seasons should each have ~450-480 played games.